In [1]:
%load_ext autoreload
%autoreload 2
import os
import csv
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

In [2]:
import random
def set_seed(seed): 
    torch.backends.cudnn.deterministic = True 
    torch.backends.cudnn.benchmark = False 
    torch.manual_seed(seed) 
    torch.cuda.manual_seed_all(seed) 
    np.random.seed(seed) 
    random.seed(seed)
    
set_seed(42)

In [3]:
from file_loader import get_all_files
data = 'data/good_images/'
files = get_all_files(data, 'png')
good_image_names =[]
for file in files:
    out_dir, img_name = os.path.split(file)
    _, sample_folder = os.path.split(out_dir)
    good_image_names.append(sample_folder+'/'+img_name)

In [4]:
len(good_image_names)

12064

In [5]:
data_df = pd.read_csv('data/all_data_new.csv')
clean_df = data_df.loc[data_df['image_name'].isin(good_image_names)].reset_index(drop = True)  #removing badly segmented crops
print(len(clean_df))
filt = (clean_df['artifact']!='ART')
clean_df = clean_df.loc[filt].reset_index(drop = True)           # removing artifacts (173)
clean_df = clean_df.drop(columns =['artifact'])
clean_df['hemo_dist'].replace({'HYO1':'HYPO', 'HYO2':'HYPO', 'HYO3':'HYPO', 'HYO4':'HYPO', \
                               'HYR1':'HYPR', 'HYR2':'HYPR', 'HYR3':'HYPR', 'HYR4':'HYPR'}, inplace=True)
clean_df.to_csv('data/all_data_new_clean1.csv', index=False)
print(len(clean_df))

12064
11891


#### Data which was labelled

In [6]:
data_df = pd.read_csv('data/all_data_new.csv')
print(data_df['size'].value_counts())
print("----------------------------------")
print(data_df['shape'].value_counts())
print("----------------------------------")
print(data_df['hemo_dist'].value_counts())
print("----------------------------------")
print(data_df['inclusion'].value_counts())
print("----------------------------------")
print(data_df['artifact'].value_counts())

NORM    11709
MICR     1309
NONE      461
MACR      250
Name: size, dtype: int64
----------------------------------
NONE    11102
ECHI      770
OVAL      632
TEAR      484
SCHI      169
ACAN      150
HELM      142
ELLI      118
SICK       83
SPHE       64
BITE       15
Name: shape, dtype: int64
----------------------------------
NONE    11469
HYO1      500
TARG      451
HYR1      370
HYO3      278
HYO2      260
STOM      140
HYO4      137
HYR2       52
HYR3       40
HYR4       32
Name: hemo_dist, dtype: int64
----------------------------------
NONE                12558
RETI                  401
MALA                  302
HOJO                  139
BAST, RETI            108
NRBC                   87
PABO                   65
BAST                   45
BAST, NRBC             10
BAST, HOJO              4
BAST, HOJO, RETI        4
HOJO, RETI              3
PABO, RETI              2
HOJO, NRBC              1
Name: inclusion, dtype: int64
----------------------------------
NONE    13268
ART    

#### Data free from bad images and artifacts

In [7]:
data_df_clean = pd.read_csv('data/all_data_new_clean1.csv')
print(data_df_clean['size'].value_counts())
print("----------------------------------")
print(data_df_clean['shape'].value_counts())
print("----------------------------------")
print(data_df_clean['hemo_dist'].value_counts())
print("----------------------------------")
print(data_df_clean['inclusion'].value_counts())

NORM    10460
MICR     1235
MACR      196
Name: size, dtype: int64
----------------------------------
NONE    9540
ECHI     711
OVAL     587
TEAR     442
SCHI     141
HELM     126
ACAN     122
ELLI     107
SPHE      59
SICK      42
BITE      14
Name: shape, dtype: int64
----------------------------------
NONE    9913
HYPO    1040
HYPR     447
TARG     380
STOM     111
Name: hemo_dist, dtype: int64
----------------------------------
NONE                11197
RETI                  273
MALA                  164
HOJO                   93
BAST, RETI             67
BAST                   33
PABO                   31
NRBC                   23
BAST, HOJO              3
HOJO, RETI              3
BAST, HOJO, RETI        3
PABO, RETI              1
Name: inclusion, dtype: int64


#### Splitting data to train and test csv files and distributions of train labels

In [8]:
data_df_clean = pd.read_csv('data/all_data_new_clean1.csv')
train_inds, test_inds = train_test_split(np.array(list(range(data_df_clean.shape[0]))), test_size=0.2, random_state=2)
train_df = data_df_clean.iloc[train_inds,:].reset_index(drop=True)
test_df = data_df_clean.iloc[test_inds,:].reset_index(drop=True)

train_df.to_csv('data/train_new_clean.csv', index=False)
test_df.to_csv('data/test_new_clean.csv', index =False)

print(train_df['size'].value_counts())
print("----------------------------------")
print(train_df['shape'].value_counts())
print("----------------------------------")
print(train_df['hemo_dist'].value_counts())
print("----------------------------------")
print(train_df['inclusion'].value_counts())

NORM    8369
MICR     986
MACR     157
Name: size, dtype: int64
----------------------------------
NONE    7655
ECHI     561
OVAL     460
TEAR     355
SCHI     108
HELM     103
ACAN      92
ELLI      84
SPHE      48
SICK      35
BITE      11
Name: shape, dtype: int64
----------------------------------
NONE    7953
HYPO     818
HYPR     341
TARG     305
STOM      95
Name: hemo_dist, dtype: int64
----------------------------------
NONE                8945
RETI                 223
MALA                 139
HOJO                  81
BAST, RETI            49
BAST                  28
PABO                  25
NRBC                  16
BAST, HOJO             2
HOJO, RETI             2
BAST, HOJO, RETI       2
Name: inclusion, dtype: int64


##### Undersampling 50% of NORMAL crops     (2470 normal crops out of 4940 normal crops removed)

In [9]:
norm_samples = np.where((train_df['size']=='NORM')&(train_df['shape']=='NONE')&(train_df['hemo_dist']=='NONE')&(train_df['inclusion']=='NONE'))[0]
print(len(norm_samples))
drop_size = int(0.5*len(norm_samples))
drop_ind = np.random.choice(norm_samples, size = drop_size, replace = False)
train_df_undersampled = train_df.drop(train_df.index[list(drop_ind)])
train_df_undersampled = train_df_undersampled.sample(frac=1).reset_index(drop=True)

4940


In [10]:
print(train_df_undersampled['size'].value_counts())
print("----------------------------------")
print(train_df_undersampled['shape'].value_counts())
print("----------------------------------")
print(train_df_undersampled['hemo_dist'].value_counts())
print("----------------------------------")
print(train_df_undersampled['inclusion'].value_counts())

NORM    5899
MICR     986
MACR     157
Name: size, dtype: int64
----------------------------------
NONE    5185
ECHI     561
OVAL     460
TEAR     355
SCHI     108
HELM     103
ACAN      92
ELLI      84
SPHE      48
SICK      35
BITE      11
Name: shape, dtype: int64
----------------------------------
NONE    5483
HYPO     818
HYPR     341
TARG     305
STOM      95
Name: hemo_dist, dtype: int64
----------------------------------
NONE                6475
RETI                 223
MALA                 139
HOJO                  81
BAST, RETI            49
BAST                  28
PABO                  25
NRBC                  16
BAST, HOJO             2
HOJO, RETI             2
BAST, HOJO, RETI       2
Name: inclusion, dtype: int64


In [11]:
df_copy = train_df_undersampled
min_samples = 200

size_rare_list = ['MACR']

for item in size_rare_list:
    size_samples = np.where((df_copy['size']==item))[0]
    len_samples = len(df_copy[df_copy['size']==item])
    gap_num = min_samples - len_samples
    temp_df = df_copy.iloc[np.random.choice(size_samples, size = gap_num)]
    df_copy = df_copy.append(temp_df, ignore_index = True)


df_copy = df_copy.sample(frac=1).reset_index(drop=True)
print(df_copy['size'].value_counts())
print("----------------------------------")
print(df_copy['shape'].value_counts())
print("----------------------------------")
print(df_copy['hemo_dist'].value_counts())
print("----------------------------------")
print(df_copy['inclusion'].value_counts())

NORM    5899
MICR     986
MACR     200
Name: size, dtype: int64
----------------------------------
NONE    5225
ECHI     561
OVAL     463
TEAR     355
SCHI     108
HELM     103
ACAN      92
ELLI      84
SPHE      48
SICK      35
BITE      11
Name: shape, dtype: int64
----------------------------------
NONE    5505
HYPO     827
HYPR     350
TARG     308
STOM      95
Name: hemo_dist, dtype: int64
----------------------------------
NONE                6515
RETI                 225
MALA                 139
HOJO                  81
BAST, RETI            49
BAST                  29
PABO                  25
NRBC                  16
BAST, HOJO             2
HOJO, RETI             2
BAST, HOJO, RETI       2
Name: inclusion, dtype: int64


In [12]:
shape_rare_list = ['ELLI','HELM','SPHE','SCHI','SICK','BITE','ACAN']

for item in shape_rare_list:
    shape_samples = np.where((df_copy['shape']==item))[0]
    len_samples = len(df_copy[df_copy['shape']==item])
    gap_num = min_samples - len_samples
    temp_df = df_copy.iloc[np.random.choice(shape_samples, size = gap_num)]
    df_copy = df_copy.append(temp_df, ignore_index = True)


df_copy = df_copy.sample(frac=1).reset_index(drop=True)
print(df_copy['size'].value_counts())
print("----------------------------------")
print(df_copy['shape'].value_counts())
print("----------------------------------")
print(df_copy['hemo_dist'].value_counts())
print("----------------------------------")
print(df_copy['inclusion'].value_counts())

NORM    6565
MICR    1239
MACR     200
Name: size, dtype: int64
----------------------------------
NONE    5225
ECHI     561
OVAL     463
TEAR     355
HELM     200
BITE     200
SPHE     200
ELLI     200
SCHI     200
ACAN     200
SICK     200
Name: shape, dtype: int64
----------------------------------
NONE    6410
HYPO     841
HYPR     350
TARG     308
STOM      95
Name: hemo_dist, dtype: int64
----------------------------------
NONE                7424
RETI                 234
MALA                 139
HOJO                  81
BAST, RETI            49
BAST                  29
PABO                  25
NRBC                  16
BAST, HOJO             3
HOJO, RETI             2
BAST, HOJO, RETI       2
Name: inclusion, dtype: int64


In [13]:
hemo_rare_list = ['STOM']

for item in hemo_rare_list:
    hemo_samples = np.where((df_copy['hemo_dist']==item))[0]
    len_samples = len(df_copy[df_copy['hemo_dist']==item])
    gap_num = min_samples - len_samples
    temp_df = df_copy.iloc[np.random.choice(hemo_samples, size = gap_num)]
    df_copy = df_copy.append(temp_df, ignore_index = True)


df_copy = df_copy.sample(frac=1).reset_index(drop=True)
print(df_copy['size'].value_counts())
print("----------------------------------")
print(df_copy['shape'].value_counts())
print("----------------------------------")
print(df_copy['hemo_dist'].value_counts())
print("----------------------------------")
print(df_copy['inclusion'].value_counts())

NORM    6668
MICR    1239
MACR     202
Name: size, dtype: int64
----------------------------------
NONE    5330
ECHI     561
OVAL     463
TEAR     355
HELM     200
BITE     200
SPHE     200
ELLI     200
SCHI     200
ACAN     200
SICK     200
Name: shape, dtype: int64
----------------------------------
NONE    6410
HYPO     841
HYPR     350
TARG     308
STOM     200
Name: hemo_dist, dtype: int64
----------------------------------
NONE                7528
RETI                 235
MALA                 139
HOJO                  81
BAST, RETI            49
BAST                  29
PABO                  25
NRBC                  16
BAST, HOJO             3
HOJO, RETI             2
BAST, HOJO, RETI       2
Name: inclusion, dtype: int64


In [14]:
inclusion_rare_list = ['RETI','MALA','HOJO','BAST, RETI','NRBC','PABO','BAST','BAST, HOJO','BAST, HOJO, RETI','HOJO, RETI']
min_samples = 400
for item in inclusion_rare_list:
    inclusion_samples = np.where((df_copy['inclusion']==item))[0]
    len_samples = len(df_copy[df_copy['inclusion']==item])
    gap_num = min_samples - len_samples
    temp_df = df_copy.iloc[np.random.choice(inclusion_samples, size = gap_num)]
    df_copy = df_copy.append(temp_df, ignore_index = True)


df_copy = df_copy.sample(frac=1).reset_index(drop=True)
print(df_copy['size'].value_counts())
print("----------------------------------")
print(df_copy['shape'].value_counts())
print("----------------------------------")
print(df_copy['hemo_dist'].value_counts())
print("----------------------------------")
print(df_copy['inclusion'].value_counts())


NORM    9597
MICR    1680
MACR     251
Name: size, dtype: int64
----------------------------------
NONE    8051
OVAL     861
ECHI     575
HELM     474
TEAR     356
SICK     211
BITE     200
SCHI     200
SPHE     200
ELLI     200
ACAN     200
Name: shape, dtype: int64
----------------------------------
NONE    9642
HYPO     953
HYPR     398
TARG     334
STOM     201
Name: hemo_dist, dtype: int64
----------------------------------
NONE                7528
BAST, HOJO           400
HOJO, RETI           400
NRBC                 400
RETI                 400
BAST                 400
PABO                 400
MALA                 400
BAST, RETI           400
BAST, HOJO, RETI     400
HOJO                 400
Name: inclusion, dtype: int64


In [15]:
train_df_processed = df_copy
train_df_processed.to_csv('data/train_processed_new_clean1.csv',index=False)

In [16]:
processed_df = pd.read_csv('data/train_processed_new_clean1.csv')
processed_df

,image_name,size,shape,hemo_dist,inclusion
0,EH5_Siemens012/jai_0000059_440_1054.png,NORM,NONE,TARG,NONE
1,EH5_Siemens012/jai_0001353_1286_1185.png,NORM,HELM,NONE,"BAST, HOJO"
2,EH5_Siemens012/jai_0001474_210_726.png,MICR,NONE,NONE,BAST
3,EH5_Siemens012/jai_0000182_530_1077.png,NORM,NONE,NONE,NRBC
4,CDB_Sample030/jai_0000017_642_109.png,NORM,NONE,NONE,HOJO
...,...,...,...,...,...
11523,EH5_Siemens003/jai_0000980_1447_1154.png,NORM,NONE,NONE,NONE
11524,EH5_Siemens013/jai_0000413_929_296.png,NORM,TEAR,NONE,NONE
11525,EH5_Siemens083/jai_0001430_74_521.png,NORM,OVAL,NONE,NONE
11526,CDB_Sample125/jai_0000274_137_428.png,NORM,ECHI,NONE,NONE


In [17]:
print(test_df['size'].value_counts())
print("----------------------------------")
print(test_df['shape'].value_counts())
print("----------------------------------")
print(test_df['hemo_dist'].value_counts())
print("----------------------------------")
print(test_df['inclusion'].value_counts())

NORM    2091
MICR     249
MACR      39
Name: size, dtype: int64
----------------------------------
NONE    1885
ECHI     150
OVAL     127
TEAR      87
SCHI      33
ACAN      30
ELLI      23
HELM      23
SPHE      11
SICK       7
BITE       3
Name: shape, dtype: int64
----------------------------------
NONE    1960
HYPO     222
HYPR     106
TARG      75
STOM      16
Name: hemo_dist, dtype: int64
----------------------------------
NONE                2252
RETI                  50
MALA                  25
BAST, RETI            18
HOJO                  12
NRBC                   7
PABO                   6
BAST                   5
PABO, RETI             1
HOJO, RETI             1
BAST, HOJO             1
BAST, HOJO, RETI       1
Name: inclusion, dtype: int64
